In [5]:
import json, os

In [6]:
CACHE_FILE  = r"E:\Callproject\assembled2\#out\_file_cache.json"
JS_OUT_PATH = r"E:\Callproject\assembled2\#out\batch_export.js"
LOG_PATH    = r"E:\Callproject\assembled2\#out\_batch_export_js.log"

with open(CACHE_FILE, "r", encoding="utf-8") as f:
    qxp_files = json.load(f)

# Escape backslashes for embedding in a JS string literal
def js_str(path):
    return path.replace("\\", "\\\\")

file_array = ",\n    ".join(f'"{js_str(p)}"' for p in qxp_files)

js = f"""
// batch_export.js  —  auto-generated by Python, run once from QuarkXPress Scripts menu
// Files: {len(qxp_files)}

function extractText(dom) {{
    var result = {{
        layout_name: dom.getAttribute("layout-name"),
        boxes: []
    }};

    var boxes = dom.querySelectorAll("qx-box[box-content-type='text']");
    for (var i = 0; i < boxes.length; i++) {{
        var box = boxes[i];
        var boxData = {{
            box_id: box.getAttribute("box-id"),
            paragraphs: []
        }};

        var paras = box.querySelectorAll("qx-p");
        for (var j = 0; j < paras.length; j++) {{
            var text = paras[j].textContent.trim();
            if (text.length > 0) {{
                boxData.paragraphs.push(text);
            }}
        }}

        if (boxData.paragraphs.length > 0) {{
            result.boxes.push(boxData);
        }}
    }}

    return result;
}}

var files = [
    {file_array}
];

var succeeded = 0;
var failed    = 0;

for (var i = 0; i < files.length; i++) {{
    var qxpPath = files[i];
    var stem    = qxpPath.replace(/_9x\\.qxp$/i, "");
    var pdfPath = stem + ".pdf";
    var jsonPath= stem + ".json";

    try {{
        var proj   = app.openProject(qxpPath, 0);
        var layout = proj.getLayoutByIndex(0);

        // PDF export
        layout.exportLayoutAsPDF(pdfPath);

        // Text extraction
        var dom      = app.activeLayoutDOM();
        var textData = extractText(dom);
        textData.source_file = qxpPath;
        fs.writeFileSync(jsonPath, JSON.stringify(textData, null, 2));

        proj.closeProject(false);
        succeeded++;

    }} catch (e) {{
        fs.writeFileSync(stem + "_ERROR.txt", e.toString());
        failed++;
        try {{ app.activeProject().closeProject(false); }} catch(e2) {{}}
    }}
}}

// Write summary log
var summary = {{
    total:     files.length,
    succeeded: succeeded,
    failed:    failed
}};
fs.writeFileSync("{js_str(LOG_PATH)}", JSON.stringify(summary, null, 2));
"""

with open(JS_OUT_PATH, "w", encoding="utf-8") as f:
    f.write(js)

print(f"JS script written to: {JS_OUT_PATH}")
print(f"Covers {len(qxp_files)} files.")

JS script written to: E:\Callproject\assembled2\#out\batch_export.js
Covers 68785 files.
